In [ ]:
import argparse
import glob
import hashlib
import io
import math
import os
import random

import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from PIL import Image, ImageEnhance, ImageFilter



In [ ]:


def cbr(i, o):
    return nn.Sequential(nn.Conv2d(i, o, 3, padding=1), nn.BatchNorm2d(o), nn.ReLU(True))


class Encoder(nn.Module):
    def __init__(self, L, ch=64):
        super().__init__()
        self.feat = nn.Sequential(cbr(3, ch), cbr(ch, ch), cbr(ch, ch), cbr(ch, ch))
        self.mix = nn.Sequential(cbr(ch + L + 3, ch), nn.Conv2d(ch, 3, 1))

    def forward(self, x, m):
        # x: [B,3,H,W] in [0,1],  m: [B,L] in {0,1}
        B, _, H, W = x.shape
        f = self.feat(x - 0.5)
        mm = (m * 2 - 1).view(B, -1, 1, 1).expand(-1, -1, H, W)
        residual = self.mix(torch.cat([f, mm, x - 0.5], 1))
        return x + residual


class Decoder(nn.Module):
    def __init__(self, L, ch=64):
        super().__init__()
        self.net = nn.Sequential(cbr(3, ch), *[cbr(ch, ch) for _ in range(6)], cbr(ch, L))
        self.fc = nn.Linear(L, L)

    def forward(self, x):
        y = self.net(x - 0.5).mean(dim=(2, 3))  
        return self.fc(y)  # logits


class Discriminator(nn.Module):
    def __init__(self, ch=64):
        super().__init__()
        self.net = nn.Sequential(cbr(3, ch), cbr(ch, ch), cbr(ch, ch))
        self.fc = nn.Linear(ch, 1)

    def forward(self, x):
        return self.fc(self.net(x - 0.5).mean(dim=(2, 3)))




In [ ]:


def ste_round(x):
    return x + (x.round() - x).detach()  # straight-through estimator


def _dct_matrix():
    D = torch.zeros(8, 8)
    for k in range(8):
        for n in range(8):
            a = math.sqrt(1 / 8) if k == 0 else math.sqrt(2 / 8)
            D[k, n] = a * math.cos((2 * n + 1) * k * math.pi / 16)
    return D


DCT = _dct_matrix()
Q50 = torch.tensor([
    [16, 11, 10, 16, 24, 40, 51, 61],
    [12, 12, 14, 19, 26, 58, 60, 55],
    [14, 13, 16, 24, 40, 57, 69, 56],
    [14, 17, 22, 29, 51, 87, 80, 62],
    [18, 22, 37, 56, 68, 109, 103, 77],
    [24, 35, 55, 64, 81, 104, 113, 92],
    [49, 64, 78, 87, 103, 121, 120, 101],
    [72, 92, 95, 98, 112, 100, 103, 99]], dtype=torch.float32)


def qtable(q):
    s = 5000 / q if q < 50 else 200 - 2 * q
    return torch.clamp(torch.floor((Q50 * s + 50) / 100), 1, 255)





In [ ]:
def jpeg_diff(x, q):
    H8, W8 = H // 8 * 8, W // 8 * 8
    y = x[:, :, :H8, :W8] * 255 - 128
    b = y.reshape(B, C, H8 // 8, 8, W8 // 8, 8).permute(0, 1, 2, 4, 3, 5)
    D = DCT.to(x)
    Q = qtable(q).to(x)
    c = D @ b @ D.T
    c = ste_round(c / Q) * Q
    b = D.T @ c @ D
    y = b.permute(0, 1, 2, 4, 3, 5).reshape(B, C, H8, W8)
    out = (y + 128) / 255
    if (H8, W8) != (H, W):
        out = F.pad(out, (0, W - W8, 0, H - H8), mode="replicate")
    return out


def gblur(x, sigma):
    k = int(2 * math.ceil(2 * sigma) + 1)
    ax = torch.arange(k, dtype=x.dtype, device=x.device) - k // 2
    g = torch.exp(-ax ** 2 / (2 * sigma ** 2))
    g = g / g.sum()
    w = (g[:, None] * g[None, :]).expand(3, 1, k, k).contiguous()
    return F.conv2d(F.pad(x, (k // 2,) * 4, mode="reflect"), w, groups=3)


def crop_resize(x, scale):
    B, C, H, W = x.shape
    ch, cw = max(8, int(H * scale)), max(8, int(W * scale))
    t = random.randint(0, H - ch)
    l = random.randint(0, W - cw)
    return F.interpolate(x[:, :, t:t + ch, l:l + cw], size=(H, W), mode="bilinear", align_corners=False)


def down_up(x, f):
    B, C, H, W = x.shape
    s = F.interpolate(x, size=(max(8, int(H * f)), max(8, int(W * f))), mode="bilinear", align_corners=False)
    return F.interpolate(s, size=(H, W), mode="bilinear", align_corners=False)




In [ ]:

def random_attack(xe, cover):
    choice = random.choice(["none", "noise", "blur", "dropout", "crop", "resize", "jpeg", "jpeg"])
    if choice == "none":
        return x
    if choice == "noise":
        return x + torch.randn_like(x) * random.uniform(0.005, 0.03)
    if choice == "blur":
        return gblur(x, random.uniform(0.5, 1.5))
    if choice == "dropout":
        mask = (torch.rand(x.shape[0], 1, *x.shape[2:], device=x.device) > random.uniform(0.1, 0.4)).float()
        return x * mask + cover * (1 - mask)
    if choice == "crop":
        return crop_resize(x, random.uniform(0.6, 0.95))
    if choice == "resize":
        return down_up(x, random.uniform(0.4, 0.9))
    return jpeg_diff(x, random.randint(30, 90))




In [ ]:


class ImgFolder(torch.utils.data.Dataset):
    def __init__(self, folder, size, repeat=1):
        exts = ("jpg", "jpeg", "png", "bmp", "webp")
        self.files = [p for e in exts for p in glob.glob(os.path.join(folder, "**", f"*.{e}"), recursive=True)]
        self.files += [p for e in exts for p in glob.glob(os.path.join(folder, "**", f"*.{e.upper()}"), recursive=True)]
        self.files = sorted(set(self.files))
        if not self.files:
            raise SystemExit(f"'{folder}' blank")
        self.size, self.repeat = size, repeat

    def __len__(self):
        return len(self.files) * self.repeat

    def __getitem__(self, i):
        img = Image.open(self.files[i % len(self.files)]).convert("RGB")
        w, h = img.size
        if min(w, h) < self.size:
            f = self.size / min(w, h)
            img = img.resize((math.ceil(w * f), math.ceil(h * f)), Image.BICUBIC)
            w, h = img.size
        l, t = random.randint(0, w - self.size), random.randint(0, h - self.size)
        img = img.crop((l, t, l + self.size, t + self.size))
        if random.random() < 0.5:
            img = img.transpose(Image.FLIP_LEFT_RIGHT)
        return torch.from_numpy(np.asarray(img).copy()).permute(2, 0, 1).float() / 255




In [ ]:


def get_device(name):
    if name != "auto":
        return torch.device(name)
    return torch.device("cuda" if torch.cuda.is_available() else "cpu")


def psnr(a, b):
    mse = torch.mean((a - b) ** 2).item()
    return 99.0 if mse == 0 else 10 * math.log10(1 / mse)


def id_to_bits(id_str, key, L):
    h = hashlib.sha256(f"{key}|{id_str}".encode()).digest()
    return np.unpackbits(np.frombuffer(h, dtype=np.uint8))[:L].astype(np.float32)





In [ ]:

def to_tensor(img, dev):
    return torch.from_numpy(np.asarray(img.convert("RGB")).copy()).permute(2, 0, 1).float().div(255).unsqueeze(0).to(dev)


def to_pil(t):
    a = (torch.clamp(t[0], 0, 1) * 255).round().byte().permute(1, 2, 0).cpu().numpy()
    return Image.fromarray(a)


def load_ckpt(path, dev):
    ck = torch.load(path, map_location=dev)
    L, ch = ck["L"], ck["ch"]
    enc, dec = Encoder(L, ch).to(dev), Decoder(L, ch).to(dev)
    enc.load_state_dict(ck["enc"])
    dec.load_state_dict(ck["dec"])
    return enc.eval(), dec.eval(), L


@torch.no_grad()
def embed_pil(enc, img, bits, dev):
    x = to_tensor(img, dev)
    m = torch.from_numpy(bits).float().unsqueeze(0).to(dev)
    return to_pil(enc(x, m))


@torch.no_grad()
def decode_pil(dec, img, dev):
    return (torch.sigmoid(dec(to_tensor(img, dev)))[0] > 0.5).float().cpu().numpy()




In [ ]:

def cmd_train(a):
    dev = get_device(a.device)
    print("Device:", dev)
    ds = ImgFolder(a.data, a.size, a.repeat)
    print(f"{len(ds.files)} images, {len(ds)} samples per epoch")

    dl = torch.utils.data.DataLoader(
        ds,
        batch_size=a.batch,
        shuffle=True,
        drop_last=True,
        num_workers=a.workers
    )

    enc, dec, dis = (
        Encoder(a.L, a.ch).to(dev),
        Decoder(a.L, a.ch).to(dev),
        Discriminator(a.ch).to(dev)
    )

    opt_g = torch.optim.Adam(
        list(enc.parameters()) + list(dec.parameters()),
        lr=a.lr
    )
    opt_d = torch.optim.Adam(dis.parameters(), lr=a.lr)
    bce = nn.BCEWithLogitsLoss()

    for ep in range(1, a.epochs + 1):
        enc.train()
        dec.train()
        dis.train()

        tot = {"acc": 0, "psnr": 0, "n": 0}

        for x in dl:
            x = x.to(dev)
            B = x.shape[0]

            m = torch.randint(0, 2, (B, a.L), device=dev).float()
            ones = torch.ones(B, 1, device=dev)
            zeros = torch.zeros(B, 1, device=dev)

            # Train Discriminator
            with torch.no_grad():
                xe = enc(x, m)

            d_loss = bce(dis(x), ones) + bce(dis(xe), zeros)

            opt_d.zero_grad()
            d_loss.backward()
            opt_d.step()

            # Train Encoder and Decoder
            xe = enc(x, m)
            logits = dec(random_attack(xe, x))

            loss = (
                a.w_img * F.mse_loss(xe, x)
                + a.w_msg * bce(logits, m)
                + a.w_adv * bce(dis(xe), ones)
            )

            opt_g.zero_grad()
            loss.backward()
            opt_g.step()

            with torch.no_grad():
                tot["acc"] += (
                    ((logits > 0).float() == m)
                    .float()
                    .mean()
                    .item()
                )
                tot["psnr"] += psnr(torch.clamp(xe, 0, 1), x)
                tot["n"] += 1

        n = tot["n"]

        print(
            f"Epoch {ep:3d}/{a.epochs}  "
            f"Bit Accuracy (with attacks)={tot['acc']/n:.3f}  "
            f"PSNR={tot['psnr']/n:.1f} dB"
        )

        torch.save(
            {
                "enc": enc.state_dict(),
                "dec": dec.state_dict(),
                "L": a.L,
                "ch": a.ch
            },
            a.out
        )

    print("Model saved to:", a.out)


In [ ]:

def cmd_embed(a):
    dev = get_device(a.device)
    enc, _, L = load_ckpt(a.ckpt, dev)

    img = Image.open(a.inp).convert("RGB")
    out = embed_pil(enc, img, id_to_bits(a.id, a.key, L), dev)
    out.save(a.out, format="PNG")

    p = 10 * math.log10(
        255 ** 2 / max(
            1e-9,
            np.mean(
                (np.asarray(img, float) - np.asarray(out, float)) ** 2
            )
        )
    )

    print(f"Saved: {a.out}   PSNR={p:.1f} dB   Bits={L}")


def cmd_extract(a):
    dev = get_device(a.device)
    _, dec, L = load_ckpt(a.ckpt, dev)

    bits = decode_pil(dec, Image.open(a.inp), dev)

    print("Extracted bits:", "".join(str(int(b)) for b in bits))

    if a.id:
        exp = id_to_bits(a.id, a.key, L)
        matches = int((bits == exp).sum())
        p = binom_pvalue(matches, L)

        print(
            f"Matched {matches}/{L} bits  "
            f"(probability of matching this many bits by chance ≈ {p:.2e})"
        )

        print(
            "Result:",
            "Watermark detected ✅" if p < 1e-6
            else "Inconclusive / Not detected ❌"
        )


In [ ]:

def _jpeg(q):
    def f(im):
        b = io.BytesIO()
        im.save(b, "JPEG", quality=q)
        b.seek(0)
        return Image.open(b).convert("RGB")
    return f


def _resize(f):
    return lambda im: im.resize(
        (
            max(8, int(im.width * f)),
            max(8, int(im.height * f))
        ),
        Image.BILINEAR
    ).resize(im.size, Image.BILINEAR)


def _crop(f):
    def g(im):
        w, h = int(im.width * f), int(im.height * f)
        l, t = (im.width - w) // 2, (im.height - h) // 2
        return im.crop(
            (l, t, l + w, t + h)
        ).resize(im.size, Image.BILINEAR)
    return g


def _noise(std):
    def f(im):
        a = (
            np.asarray(im, np.float32)
            + np.random.randn(im.height, im.width, 3) * std
        )
        return Image.fromarray(np.clip(a, 0, 255).astype(np.uint8))
    return f


ATTACKS = {
    "No attack": lambda im: im,
    "JPEG q=90": _jpeg(90),
    "JPEG q=70": _jpeg(70),
    "JPEG q=50": _jpeg(50),
    "JPEG q=30": _jpeg(30),
    "Resize 0.7x": _resize(0.7),
    "Resize 0.5x": _resize(0.5),
    "Crop 80%": _crop(0.8),
    "Crop 60%": _crop(0.6),
    "Blur r=1": lambda im: im.filter(ImageFilter.GaussianBlur(1)),
    "Noise sigma=5": _noise(5),
    "Noise sigma=10": _noise(10),
    "Brightness 1.2x (unseen)": lambda im: ImageEnhance.Brightness(im).enhance(1.2),
    "Rotation 5 deg (unseen)": lambda im: im.rotate(
        5, resample=Image.BILINEAR
    ),
}


def cmd_test(a):
    dev = get_device(a.device)
    enc, dec, L = load_ckpt(a.ckpt, dev)
    ds = ImgFolder(a.data, 8)

    res = {k: [] for k in ATTACKS}
    psnrs = []

    for i, path in enumerate(ds.files[: a.max_images]):
        img = Image.open(path).convert("RGB")

        s = a.max_side / max(img.size)
        if s < 1:
            img = img.resize(
                (int(img.width * s), int(img.height * s)),
                Image.LANCZOS
            )

        bits = id_to_bits(f"img{i}", a.key, L)
        wm = embed_pil(enc, img, bits, dev)

        psnrs.append(
            10 * math.log10(
                255 ** 2 / max(
                    1e-9,
                    np.mean(
                        (
                            np.asarray(img, float)
                            - np.asarray(wm, float)
                        ) ** 2
                    )
                )
            )
        )

        for name, fn in ATTACKS.items():
            got = decode_pil(dec, fn(wm), dev)
            res[name].append(int((got == bits).sum()))

    print(
        f"\nImages: {len(psnrs)}   "
        f"Average PSNR = {np.mean(psnrs):.1f} dB   "
        f"Bits = {L}\n"
    )

    print(f"{'Attack':<26}{'Avg. bit acc.':>12}{'Detection rate':>16}")

    for name, m in res.items():
        acc = np.mean(m) / L
        det = np.mean([binom_pvalue(x, L) < 1e-6 for x in m])

        print(f"{name:<26}{acc:>12.3f}{det:>15.0%}")

    print(
        "\n(Detection = probability of a random match < 1e-6)"
    )


In [ ]:

def main():
    p = argparse.ArgumentParser()
    sub = p.add_subparsers(dest="cmd", required=True)

    common = argparse.ArgumentParser(add_help=False)
    common.add_argument("--device", default="auto")

    # Train command
    t = sub.add_parser("train", parents=[common])
    t.add_argument("--data", required=True)
    t.add_argument("--out", default="ckpt.pt")
    t.add_argument("--size", type=int, default=128)
    t.add_argument("--L", type=int, default=30, help="Number of bits")
    t.add_argument(
        "--ch",
        type=int,
        default=64,
        help="Number of channels (use 32 if training is slow on CPU)"
    )
    t.add_argument("--epochs", type=int, default=30)
    t.add_argument("--batch", type=int, default=16)
    t.add_argument(
        "--repeat",
        type=int,
        default=20,
        help="Repeat samples to increase the effective dataset size"
    )
    t.add_argument("--lr", type=float, default=1e-3)
    t.add_argument("--w_img", type=float, default=0.7)
    t.add_argument("--w_msg", type=float, default=1.0)
    t.add_argument("--w_adv", type=float, default=1e-3)
    t.add_argument("--workers", type=int, default=0)
    t.set_defaults(fn=cmd_train)

    # Embed command
    e = sub.add_parser("embed", parents=[common])
    e.add_argument("--ckpt", required=True)
    e.add_argument("--inp", required=True)
    e.add_argument("--out", required=True)
    e.add_argument("--id", required=True)
    e.add_argument("--key", default="secret")
    e.set_defaults(fn=cmd_embed)

    # Extract command
    x = sub.add_parser("extract", parents=[common])
    x.add_argument("--ckpt", required=True)
    x.add_argument("--inp", required=True)
    x.add_argument("--id", default=None)
    x.add_argument("--key", default="secret")
    x.set_defaults(fn=cmd_extract)

    # Test command
    s = sub.add_parser("test", parents=[common])
    s.add_argument("--ckpt", required=True)
    s.add_argument("--data", required=True)
    s.add_argument("--key", default="secret")
    s.add_argument("--max_images", type=int, default=100)
    s.add_argument("--max_side", type=int, default=512)
    s.set_defaults(fn=cmd_test)

    a = p.parse_args()
    a.fn(a)


if __name__ == "__main__":
    main()
